# Reproducing cuda-matmul-optimization on Colab

This notebook builds the project from a fresh clone and reruns its benchmarks:

1. Verify the GPU and CUDA toolkit.
2. Clone the repository into `/content/cuda-matmul-optimization`.
3. Build with `make ARCH=sm_75`.
4. Run the GEMM correctness and performance benchmark against cuBLAS, then the optimized MNIST inference benchmark.
5. Sweep batch sizes and chart the new results next to the ones in the README.

**Before running:** select *Runtime → Change runtime type → T4 GPU*. No GitHub account or token is needed.
Then run *Runtime → Run all*; the whole notebook takes a few minutes.

Numbers will vary somewhat between sessions with clocks, temperature, and how busy the shared GPU is.

## 1. Environment

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu or "none found")
assert gpu, "No NVIDIA GPU found: select Runtime > Change runtime type > T4 GPU"
if "T4" not in gpu:
    print("Note: the build below targets sm_75 (T4). For another GPU, change ARCH to its compute capability.")

## 2. Clone the repository

In [ ]:
import os

REPO = "/content/cuda-matmul-optimization"
RESULTS = "/content/fresh_results"  # outputs from this session, kept outside the repo

if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/loganKneller/cuda-matmul-optimization.git {REPO}

os.makedirs(RESULTS, exist_ok=True)
%cd {REPO}
!git log --oneline -3

## 3. Build

In [ ]:
!make ARCH=sm_75

In [ ]:
import shlex
import subprocess


def run(command, cwd=REPO):
    """Run a command, print its output, and fail the cell if it fails."""
    result = subprocess.run(shlex.split(command), cwd=cwd, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"{command!r} exited with status {result.returncode}")
    return result.stdout

## 4. Benchmarks

### GEMM kernels vs. cuBLAS

Runs every kernel on square matrices from 512 to 4096 and on both MLP layer shapes. Each kernel's output is checked
against cuBLAS, and the cell fails if any kernel fails. Results are written to `fresh_results/results.csv`.

In [ ]:
_ = run(f"{REPO}/gemm/gemm_bench", cwd=RESULTS)

An irregular shape that is not a multiple of any tile size, to exercise the edge-tile handling:

In [ ]:
_ = run(f"{REPO}/gemm/gemm_bench 257 129 131", cwd="/tmp")

### Optimized MNIST inference

The fully fused forward pass on all 10,000 test images. It checks every output logit against PyTorch and fails
if any differ.

In [ ]:
_ = run("./mlp_benchmark --fused")

## 5. Results and charts

### Batch-size sweep

Runs each of the four inference modes at five batch sizes (20 runs, each validated against PyTorch).

In [ ]:
import pandas as pd

columns = ["mode", "batch", "reps", "gpu_ms", "h2d_ms", "d2h_ms", "gpu_images_per_s", "e2e_images_per_s"]
rows = []
for mode in ["naive", "unfused", "fuse-layer1", "fused"]:
    for batch in [1, 64, 256, 1024, 10000]:
        output = subprocess.run(
            ["./mlp_benchmark", f"--{mode}", "--batch", str(batch)],
            cwd=REPO, capture_output=True, text=True,
        )
        if output.returncode != 0:
            raise RuntimeError(f"--{mode} --batch {batch} failed:\n{output.stdout}{output.stderr}")
        line = next(l for l in output.stdout.splitlines() if l.startswith("CSV,"))
        rows.append(line.removeprefix("CSV,").split(","))

sweep = pd.DataFrame(rows, columns=columns)
sweep[columns[1:]] = sweep[columns[1:]].apply(pd.to_numeric)
sweep.to_csv(f"{RESULTS}/batch_sweep.csv", index=False)
sweep.pivot(index="batch", columns="mode", values="gpu_ms")[["naive", "unfused", "fuse-layer1", "fused"]].round(4)

### GEMM results

In [ ]:
gemm = pd.read_csv(f"{RESULTS}/results.csv")
gemm.pivot(index="kernel", columns="shape", values="pct_of_cublas").reindex(
    ["cuBLAS", "naive", "tiled", "1d-block", "2d-block", "2d-bk16"]
)[["square-512", "square-1024", "square-2048", "square-4096", "mlp-layer1", "mlp-layer2"]].round(1)

### Charts from this session

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e6e5e0"
plt.rcParams.update({
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})
device = gemm["device"].iloc[0]

# GEMM throughput on 1024 x 1024
labels = {"naive": "Naive", "tiled": "Shared memory", "1d-block": "1D registers",
          "2d-block": "2D registers\nBK=8", "2d-bk16": "2D registers\nBK=16", "cuBLAS": "cuBLAS"}
g = gemm[gemm["shape"] == "square-1024"].set_index("kernel").loc[list(labels)]
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar([labels[k] for k in g.index], g["gflops"], width=0.62, color="#2a78d6")
for bar, gflops, pct in zip(bars, g["gflops"], g["pct_of_cublas"]):
    ax.annotate(f"{gflops:,.0f}\n({pct:.1f}%)", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", color=INK, fontsize=10)
ax.set_ylabel("Throughput (GFLOP/s)")
ax.set_ylim(0, g["gflops"].max() * 1.18)
ax.yaxis.grid(True, color=GRID)
ax.set_axisbelow(True)
ax.set_title(f"1024 × 1024 FP32 matrix multiplication · {device}", loc="left", fontweight="bold")
fig.tight_layout()
fig.savefig(f"{RESULTS}/gemm_1024.png", dpi=150)
plt.show()

# Inference latency by batch size
series = [("naive", "Naive CUDA", "#2a78d6", "o"), ("unfused", "Shape-optimized", "#eb6834", "s"),
          ("fuse-layer1", "Layer 1 fused", "#1baf7a", "^"), ("fused", "Both layers fused", "#eda100", "D")]
fig, ax = plt.subplots(figsize=(10, 4.5))
for mode, label, color, marker in series:
    s = sweep[sweep["mode"] == mode]
    ax.plot(s["batch"], s["gpu_ms"], color=color, marker=marker, markersize=7, linewidth=2, label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(sorted(sweep["batch"].unique()))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, _: ""))
ax.set_xlabel("Batch size (images)")
ax.set_ylabel("GPU-only batch latency (ms, log scale)")
ax.grid(True, which="major", color=GRID)
ax.legend(frameon=False, ncol=2)
ax.set_title(f"Inference latency by batch size · {device}", loc="left", fontweight="bold")
fig.tight_layout()
fig.savefig(f"{RESULTS}/batch_latency.png", dpi=150)
plt.show()

### Charts in the README, for comparison

These were measured in an earlier session on a T4.

In [ ]:
from IPython.display import Image, display

display(Image(f"{REPO}/assets/gemm_1024.png", width=800))
display(Image(f"{REPO}/assets/batch_latency.png", width=800))

### Download this session's results

`fresh_results/` holds `results.csv`, `batch_sweep.csv`, and both charts.

In [ ]:
!cd /content && zip -qr fresh_results.zip fresh_results && ls -lh fresh_results.zip
try:
    from google.colab import files
    files.download("/content/fresh_results.zip")
except ImportError:
    print("Not running in Colab: results are in", RESULTS)

## Optional: PyTorch baselines

The same forward pass in PyTorch on the CPU and on the GPU (cuBLAS), using the exported weights and the full
test set. This is for context only; the CUDA benchmark above does not depend on it.

In [ ]:
import statistics
import time
from pathlib import Path

import numpy as np
import torch

root = Path(REPO) / "weights/mlp"
load = lambda name, *shape: torch.from_numpy(np.fromfile(root / name, dtype=np.float32).reshape(*shape))
X = load("full_test/inputs_cuda.bin", 10000, 784)
W1, b1 = load("weights1_cuda.bin", 784, 128), load("biases1_cuda.bin", 128)
W2, b2 = load("weights2_cuda.bin", 128, 10), load("biases2_cuda.bin", 10)
reference = load("full_test/reference_logits.bin", 10000, 10)


def forward(X, W1, b1, W2, b2):
    return torch.relu(X @ W1 + b1) @ W2 + b2


def median_ms(fn, synchronize=lambda: None, warmup=5, runs=30):
    for _ in range(warmup):
        fn()
    synchronize()
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        fn()
        synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)


print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

with torch.inference_mode():
    cpu = (X, W1, b1, W2, b2)
    print("CPU matches reference:", torch.allclose(forward(*cpu), reference, atol=1e-3, rtol=1e-4))
    print(f"CPU ({torch.get_num_threads()} threads): {median_ms(lambda: forward(*cpu)):.4f} ms")

    gpu = tuple(t.cuda() for t in cpu)  # copied once, outside the timing
    print("GPU matches reference:", torch.allclose(forward(*gpu).cpu(), reference, atol=1e-3, rtol=1e-4))

    start, stop = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    for _ in range(5):
        forward(*gpu)

    torch.cuda.synchronize()
    times = []
    for _ in range(30):
        start.record()
        forward(*gpu)
        stop.record()
        stop.synchronize()
        times.append(start.elapsed_time(stop))
    print(f"PyTorch GPU: {statistics.median(times):.4f} ms (median of 30, CUDA events)")